# MATR — Online Temporal Action Localization with Memory-Augmented Transformer
### Training Notebook (Colab / Kaggle ready)

This notebook trains and evaluates **MATR** (Song, Kim, Cho & Kwak, ECCV 2024) on the **THUMOS14** dataset,
using the authors' official codebase: [`skhcjh231/MATR_codebase`](https://github.com/skhcjh231/MATR_codebase).

Paper: [arXiv:2408.02957](https://arxiv.org/abs/2408.02957) · Project page: https://skhcjh231.github.io/MATR_project/

**What this notebook does**
1. Detects the runtime (Colab / Kaggle / local) and sets up working directories accordingly (no hardcoded local paths).
2. Clones the official repository and installs its dependencies.
3. Downloads the pre-extracted THUMOS14 features + annotations + the authors' checkpoint (linked in the README).
4. Builds the MATR model, memory-augmented encoder/decoder and criterion exactly as defined in the repo.
5. Trains with GPU acceleration when available (falls back to CPU automatically), logging loss curves and mAP@[0.3:0.7].
6. Evaluates on the THUMOS14 test split and plots metrics.
7. Saves the best checkpoint.
8. Runs a final inference example that prints predicted action instances for a sample video.

> **Note on runtime:** Full training (100 epochs on THUMOS14) can take many hours even on a single high-end GPU.
> `NUM_EPOCHS` below defaults to a small demo value so the whole notebook can be smoke-tested quickly end-to-end.
> Increase it (and consult `scripts/train_ontal.sh` / `util/config.py` in the repo) for a full reproduction run.


## 1. Environment setup
Detect whether we're on Colab, Kaggle, or a local/other machine, and define a writable **root directory** — every path in this notebook is derived from it, so nothing is hardcoded to a particular machine.

In [ ]:

import os, sys, subprocess, platform

def detect_environment():
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
        return "colab"
    if os.path.exists("/kaggle/working"):
        return "kaggle"
    return "local"

ENV = detect_environment()

if ENV == "colab":
    ROOT_DIR = "/content"
elif ENV == "kaggle":
    ROOT_DIR = "/kaggle/working"
else:
    ROOT_DIR = os.path.join(os.getcwd(), "matr_workspace")

os.makedirs(ROOT_DIR, exist_ok=True)
REPO_DIR = os.path.join(ROOT_DIR, "MATR_codebase")

print(f"Detected environment : {ENV}")
print(f"Root working dir      : {ROOT_DIR}")
print(f"Repo dir               : {REPO_DIR}")
print(f"Python                 : {platform.python_version()}")


## 2. Clone the official repository

In [ ]:

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/vercelfree/moshi-matr-update.git", REPO_DIR], check=True)
else:
    print("Repo already present, skipping clone.")

os.chdir(REPO_DIR)
print("\nRepository contents:")
for f in sorted(os.listdir(REPO_DIR)):
    print(" -", f)


### 2.1 Compatibility patch
The repo was written against an older NumPy (<1.24) and still uses the deprecated scalar aliases
`np.int` / `np.float` / `np.bool` / `np.object` / `np.str`, which were **removed** in modern NumPy
(they raise `AttributeError` on current Colab/Kaggle images). Rather than downgrading NumPy — which risks
breaking ABI compatibility with the pre-installed `torch`/`opencv` builds — we patch the handful of affected
lines in the cloned source to use the builtin types directly. This changes nothing about the code's behavior,
it only removes the deprecated aliases.

In [ ]:

import re, glob

DEPRECATED_ALIASES = ["int", "float", "bool", "object", "str"]

def patch_deprecated_numpy_aliases(root_dir):
    pattern = re.compile(r"\bnp\.(" + "|".join(DEPRECATED_ALIASES) + r")\b(?!\w)")
    patched_files = []
    for path in glob.glob(os.path.join(root_dir, "**", "*.py"), recursive=True):
        with open(path, "r") as f:
            src = f.read()
        new_src, n = pattern.subn(lambda m: m.group(1), src)
        if n > 0:
            with open(path, "w") as f:
                f.write(new_src)
            patched_files.append((os.path.relpath(path, root_dir), n))
    return patched_files

patched = patch_deprecated_numpy_aliases(REPO_DIR)
if patched:
    print("Patched deprecated NumPy scalar aliases (np.int/np.float/...) in:")
    for fname, n in patched:
        print(f"  - {fname}: {n} occurrence(s)")
else:
    print("No deprecated NumPy aliases found (nothing to patch).")


## 3. Install dependencies
`requirements.txt` in this repo was exported from a conda environment and contains OS-specific `file://` paths
(conda build artifacts) that cannot be installed via `pip` on Colab/Kaggle. We install the actual **runtime**
packages the code imports (see `models/`, `criterion/`, `dataset.py`, `util/`, `Evaluation/`) using standard
PyPI versions instead, which is portable across machines.

Colab/Kaggle already ship a recent PyTorch build wired to the provided GPU driver — we keep that instead of
force-installing the older `torch==2.0.0` pin from the README, which avoids breaking CUDA/driver compatibility.
Everything else is installed to match what the code needs.

In [ ]:

import torch as _torch_check
print("Pre-installed torch:", _torch_check.__version__, "| CUDA available:", _torch_check.cuda.is_available())


In [ ]:

packages = [
    "numpy",
    "pandas",
    "h5py",
    "opencv-python-headless",
    "matplotlib",
    "wandb",
    "scipy",
    "x-transformers",
    "typeguard",
    "tqdm",
    "gdown",
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages], check=True)
print("Dependencies installed.")


In [ ]:
!pip install -q --ignore-installed --force-reinstall --no-deps matplotlib

In [ ]:

# wandb is imported by the repo's training code but we do not want an interactive API-key prompt
# in an automated notebook run -> run wandb in offline/disabled mode by default.
os.environ["WANDB_MODE"] = "disabled"


## 4. GPU check

In [ ]:

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("No GPU detected — training will run on CPU and will be very slow. "
          "On Colab: Runtime > Change runtime type > GPU. On Kaggle: Settings > Accelerator > GPU.")


## 5. Download the dataset and pretrained checkpoint
The README links a public Google Drive folder containing the pre-extracted THUMOS14 features
(`thumos_all_feature_val_V3.pickle`, `thumos_all_feature_test_V3.pickle`), the annotation file
(`thumos14_v2.json`), and the authors' trained checkpoint (`best_epoch.pth`):

https://drive.google.com/drive/folders/1-V3TZNHrhb-1pnwKZvLCw-Ga56dx1pcb

We fetch it with `gdown` into the repo's expected `data/` and `checkpoint/` folders — no credentials required
since the folder is shared publicly, and no path is hardcoded to any particular local machine.

> Google Drive occasionally rate-limits anonymous folder downloads. If the automatic download below fails,
> open the link above in a browser, download the files manually, and re-upload/place them at
> `REPO_DIR/data/` and `REPO_DIR/checkpoint/` respectively, then re-run the cell after that.

> **EPIC-Kitchens users:** the next two cells (this one and the unzip cell below) are THUMOS14-only setup
> steps. Skip both if `DATASET == "epic"` in Step 6 — EPIC's annotation JSON and per-video `.npz` feature
> files are expected to already be staged on the pod at the paths configured there.

In [ ]:
# Step 1: Install the huggingface_hub library
!pip install -q huggingface_hub

DATA_DIR = os.path.join(REPO_DIR, "data")
CKPT_DIR = os.path.join(REPO_DIR, "checkpoint")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# Step 2: Import libraries
from huggingface_hub import list_repo_files, hf_hub_download
import os
import shutil

# Step 3: Set repo info
repo_id = "Darknsu/Epic_Kitchen"  # Your repo
repo_type = "dataset"  # Use "model" if it's a model repo
download_dir = DATA_DIR

# Step 4: List all files in the repo
files = list_repo_files(repo_id=repo_id, repo_type=repo_type)
print(f"🔍 Found {len(files)} files in the repo.")

# Step 5: Download all files and save them into local folder
os.makedirs(download_dir, exist_ok=True)
downloaded_files = []

for file in files:
    # Download the file
    file_path = hf_hub_download(repo_id=repo_id, filename=file, repo_type=repo_type)

    # Preserve folder structure
    local_path = os.path.join(download_dir, file)
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    shutil.copy(file_path, local_path)

    downloaded_files.append(local_path)

print("✅ All files downloaded into:", download_dir)

In [ ]:

# DATA_DIR = os.path.join(REPO_DIR, "data")
# CKPT_DIR = os.path.join(REPO_DIR, "checkpoint")
# os.makedirs(DATA_DIR, exist_ok=True)
# os.makedirs(CKPT_DIR, exist_ok=True)

# GDRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1-V3TZNHrhb-1pnwKZvLCw-Ga56dx1pcb"

# import gdown

# def try_download_folder(url, out_dir):
#     try:
#         gdown.download_folder(url=url, output=out_dir, quiet=False, use_cookies=False)
#         return True
#     except Exception as e:
#         print("Automatic Google Drive download failed:", e)
#         print("Please download the dataset manually from:", url)
#         return False

# _download_target = os.path.join(ROOT_DIR, "matr_gdrive_download")
# os.makedirs(_download_target, exist_ok=True)
# _ok = try_download_folder(GDRIVE_FOLDER_URL, _download_target)

# if _ok:
#     import shutil
#     for fname in os.listdir(_download_target):
#         src = os.path.join(_download_target, fname)
#         if fname.endswith(".pth"):
#             shutil.move(src, os.path.join(CKPT_DIR, fname))
#         elif fname.endswith((".pickle", ".pkl", ".json")):
#             shutil.move(src, os.path.join(DATA_DIR, fname))
#     print("\nData directory:", os.listdir(DATA_DIR) if os.path.exists(DATA_DIR) else "missing")
#     print("Checkpoint directory:", os.listdir(CKPT_DIR) if os.path.exists(CKPT_DIR) else "missing")


In [ ]:

# Sanity check: verify the expected files are present before continuing.
# THUMOS14-only (this checks the specific filenames downloaded in Step 5).
if DATASET == "thumos14":
    required_data_files = [
        "thumos_all_feature_val_V3.pickle",
        "thumos_all_feature_test_V3.pickle",
        "thumos14_v2.json",
    ]
    missing = [f for f in required_data_files if not os.path.exists(os.path.join(DATA_DIR, f))]
    if missing:
        print("WARNING: the following required data files are missing:", missing)
        print("Download them manually from the README's Google Drive link and place them in:", DATA_DIR)
    else:
        print("All required dataset files are present in", DATA_DIR)

    ckpt_path = os.path.join(CKPT_DIR, "best_epoch.pth")
    print("Pretrained checkpoint found:" if os.path.exists(ckpt_path) else "Pretrained checkpoint NOT found:", ckpt_path)
else:
    print(f"Skipping THUMOS14 file check (DATASET={DATASET!r}).")


## 6. Configuration
We build the same `argparse` configuration the repo's `main.py` / `scripts/train_ontal.sh` use
(`util/config.py`), but as a plain `Namespace` so it works inside a notebook (no `sys.argv` parsing).
All paths are derived from `REPO_DIR` / `ROOT_DIR` — nothing is hardcoded.

Flags mirror `scripts/train_ontal.sh`: `--rgb --flow --reduce 1 --use_focal --make_output --use_flag`.

In [ ]:

import argparse, time

# ---------------------------------------------------------------------------
# Dataset toggle: "thumos14" (default) or "epic".
# Only the dataset-specific fields below differ between the two; every
# training hyperparameter (epochs, batch, lr/scheduler, model dims, etc.)
# stays identical for both, sourced from the same defaults as util/config.py.
# ---------------------------------------------------------------------------
DATASET = "epic"   # "thumos14" or "epic"

if DATASET == "thumos14":
    dataset_cfg = dict(
        dataset="thumos14",
        video_anno=os.path.join(DATA_DIR, "thumos14_v2.json"),
        video_len_file=os.path.join(REPO_DIR, "output", "video_len_{}.json"),
        video_feature_all_train=os.path.join(DATA_DIR, "thumos_all_feature_val_V3.pickle"),
        video_feature_all_test=os.path.join(DATA_DIR, "thumos_all_feature_test_V3.pickle"),
        ontal_label_file=os.path.join(REPO_DIR, "output", "label_{}_{}_{}_{}_{}_{}_{}.h5"),
        num_of_class=21,
        feat_dim=4096,
        rgb=True,
        flow=True,
    )
elif DATASET == "epic":
    # Point this at the directory containing the per-video EPIC .npz feature
    # files on RunPod (flat "<video_id>.npz", e.g. "P01_08.npz"). Both splits
    # read from the same directory -- split membership comes from the
    # annotation JSON's "subset" field, not folder separation.
    EPIC_FEATURE_DIR = os.path.join(DATA_DIR, "features")
    dataset_cfg = dict(
        dataset="epic",
        video_anno=os.path.join(DATA_DIR, "epic_kitchens_100_verb.json"),
        video_len_file=os.path.join(REPO_DIR, "output", "video_len_epic_{}.json"),
        video_feature_all_train=EPIC_FEATURE_DIR,
        video_feature_all_test=EPIC_FEATURE_DIR,
        ontal_label_file=os.path.join(REPO_DIR, "output", "label_epic_{}_{}_{}_{}_{}_{}_{}.h5"),
        num_of_class=98,
        feat_dim=2304,
        rgb=True,
        flow=False,
    )
else:
    raise ValueError(f"Unknown DATASET: {DATASET!r}")

args = argparse.Namespace(
    # dataset (branched above)
    **dataset_cfg,
    num_frame=64,
    num_queries=10,
    p_videos=1,
    detect_len=16,
    anti_len=16,

    # model
    dropout=0.3,
    training=True,
    hidden_dim=1024,
    ffn_dim=2048,
    e_nheads=8,
    enc_layers=3,
    d_nheads=4,
    dec_layers=5,
    pre_norm=False,
    activation="gelu",
    max_memory_len=7,
    memory_sampler="gap2",
    use_flag=True,
    flag_threshold=0.5,

    # training
    epochs=2,              # <-- DEMO VALUE. Paper / README default is 100 (see util/config.py). Increase for a full run.
    batch=8,                # <-- reduced from the default 64 for a quick, memory-friendly demo run. Restore 64 for full reproduction.
    mode="train",
    min_lr=1e-8,
    max_lr=1e-5,
    weight_decay=1e-4,
    lr_gamma=0.9,
    lr_Tup=3,
    lr_Tcycle=10,
    test_freq=1,
    save_freq=1,
    drop_rate=0.3,

    # criterion
    use_empty_weight=False,
    eos_coef=1e-8,
    use_focal=True,
    reduce=1,
    cls_threshold=0.1,
    cls_coef=1,
    flag_coef=1,
    reg_l1_coef=1,
    reg_diou_coef=1,
    reg_stcls_coef=1,

    # evaluation
    nms_threshold=0.3,

    # output
    make_output=True,
    proposal_path="proposal_{}_{}_{}",

    # wandb / misc
    wandb=False,
    code_testing=True,

    # default setting
    task="ontal",
    random_seed=52,
    save_path=None,
    load_model=False,
    model_path=None,
    train_eval_step=1,
    test_eval_step=1,

    # GPU
    device="0",
    num_workers=2,
)

time_str = time.strftime("%Y-%m-%d_%H-%M-%S", time.localtime())
exp_name = "%s_[%s]_[%s]_ON_TAL_<%s>" % ("notebook", args.task, args.dataset, time_str)
args.save_path = os.path.join(REPO_DIR, "result", exp_name)
os.makedirs(args.save_path, exist_ok=True)
os.makedirs(os.path.dirname(args.video_len_file), exist_ok=True)
os.makedirs(os.path.dirname(args.ontal_label_file), exist_ok=True)

print("Experiment save path:", args.save_path)


## 7. Load and preprocess the dataset
This uses the repo's own `THUMOS14Dataset` (`dataset.py`), which:
- reads the THUMOS14 annotation JSON and the pre-extracted RGB/Flow feature pickles,
- builds fixed-length sliding-window segments (`num_frame` per segment),
- generates per-segment classification / regression / start-position / "action-in-progress" labels
  (cached to an `.h5` file under `output/` after the first run, so re-runs are fast).

In [ ]:
!unzip /workspace/matr_workspace/matr_gdrive_download/thumos_dataset.zip -d /workspace/matr_workspace/MATR_codebase/data

In [ ]:

sys.path.insert(0, REPO_DIR)

if args.dataset == "epic":
    from dataset_epic import EPICKitchensDataset as _DatasetClass
else:
    from dataset import THUMOS14Dataset as _DatasetClass

print("Building training split...")
train_dataset = _DatasetClass(args, subset="train")
print("Building test split...")
test_dataset = _DatasetClass(args, subset="test")

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=args.batch, shuffle=False,
    num_workers=args.num_workers, pin_memory=True, drop_last=False,
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=args.batch, shuffle=False,
    num_workers=args.num_workers, pin_memory=True, drop_last=False,
)

print(f"\nTrain segments: {len(train_dataset)} | Test segments: {len(test_dataset)}")
print(f"Action classes ({len(train_dataset.label_name)}): {train_dataset.label_name}")


In [ ]:
# import glob
# print(args.ontal_label_file)  # check the path pattern
# cache_files = glob.glob(args.ontal_label_file.format("*","*","*","*","*","*","*"))
# for f in cache_files:
#     print("removing:", f)
#     os.remove(f)


## 8. Build the model, criterion, optimizer and scheduler
- `MATR` (`models/models.py`): feature-reduction layers + memory-augmented Transformer encoder/decoders
  (end decoder + start decoder) + prediction heads, exactly as described in the paper.
- `CriterionMATR` (`criterion/criterion.py`) with Hungarian matching (`criterion/matcher.py`) for the
  classification / L1 / DIoU / start-classification / in-progress-flag losses.
- Adam optimizer with a warm-up cosine-annealing schedule (`CosineAnnealingWarmUpRestarts`, `util/utils.py`),
  matching the repo's training script.

> **EPIC-Kitchens caveat:** if `args.load_model=True`, `args.model_path` must point at a checkpoint shaped
> for the *current* `DATASET` (`feat_dim`/`num_of_class` must match). Loading a THUMOS14 checkpoint while
> `DATASET == "epic"` (or vice versa) won't raise an error — the filtered load below silently drops every
> shape-mismatched key — so it would warm-start from nothing rather than failing loudly. Only set
> `load_model=True` with a checkpoint you know matches the active dataset.

In [ ]:

from models import build_model
from criterion import build_criterion
from util.utils import CosineAnnealingWarmUpRestarts, print_log

model = build_model(args)
model = torch.nn.DataParallel(model).to(device)
criterion = build_criterion(args, device)

num_params = sum(p.data.nelement() for p in model.parameters())
print_log(args.save_path, "--------------------Number of parameters--------------------")
print_log(args.save_path, f"Number of full model parameters: {num_params:,}")

optimizer = torch.optim.Adam(
    model.parameters(), lr=args.min_lr, betas=(0.9, 0.999), eps=1e-8,
    weight_decay=args.weight_decay,
)
scheduler = CosineAnnealingWarmUpRestarts(
    optimizer, T_0=args.lr_Tcycle, T_mult=1, eta_max=args.max_lr,
    T_up=args.lr_Tup, gamma=args.lr_gamma,
)

# Optionally warm-start from the authors' pretrained checkpoint (if it was downloaded in Step 5)
if args.load_model and os.path.exists(str(args.model_path)):
    checkpoint = torch.load(args.model_path, map_location=device)
    pretrained_dict = {k: v for k, v in checkpoint["state_dict"].items() if k in model.state_dict()}
    model.load_state_dict(pretrained_dict)
    print(f"Loaded pretrained weights from {args.model_path}")


## 9. Training loop
We reuse the repo's own `train_one_epoch` / `test_one_epoch` functions from `on_tal_task.py` — this is the
exact same forward/backward pass, NMS post-processing and mAP evaluation (`Evaluation/eval_detection_gentime.py`)
used by `scripts/train_ontal.sh` — and add live metric collection + plotting on top so progress is visible
directly in the notebook. The best checkpoint (highest test mAP) is saved automatically.

In [ ]:

from on_tal_task import train_one_epoch, test_one_epoch
import sys, re, glob, datetime

class _Tee:
    """Writes every message to multiple streams (e.g. notebook stdout + a log file)."""
    def __init__(self, *streams):
        self.streams = streams
    def write(self, data):
        for s in self.streams:
            s.write(data)
            s.flush()
    def flush(self):
        for s in self.streams:
            s.flush()

log_file_path = os.path.join(args.save_path, "training_log.txt")
_log_file = open(log_file_path, "a", buffering=1)
_log_file.write(f"\n===== Training run started {datetime.datetime.now().isoformat()} =====\n")

_stdout_orig, _stderr_orig = sys.stdout, sys.stderr
sys.stdout = _Tee(_stdout_orig, _log_file)
sys.stderr = _Tee(_stderr_orig, _log_file)

print(f"Logging all training output to: {log_file_path}")

# ---------------------------------------------------------------------------
# Resumable per-epoch checkpoints.
# IMPORTANT: this is a dedicated directory, distinct from the `CKPT_DIR` used in
# Step 5 for the authors' downloaded checkpoint, and distinct from args.save_path
# (which is stamped with this run's timestamp -- see Step 6, so a fresh kernel
# restart gets a brand new, empty save_path and would never see checkpoints from
# an earlier run there). RESUME_CKPT_DIR is stable across restarts so resume works.
# Suffixed by args.dataset (except for the original "thumos14" path, kept
# unprefixed for backward compatibility) so switching DATASET in the config
# cell can never load a checkpoint shaped for the other dataset.
# ---------------------------------------------------------------------------
RESUME_CKPT_DIR = os.path.join(REPO_DIR, "checkpoints_ontal" if args.dataset == "thumos14" else f"checkpoints_ontal_{args.dataset}")
os.makedirs(RESUME_CKPT_DIR, exist_ok=True)

def _find_latest_checkpoint(ckpt_dir):
    latest_epoch, latest_path = 0, None
    for path in glob.glob(os.path.join(ckpt_dir, "epoch*.pt")):
        m = re.fullmatch(r"epoch(\d+)\.pt", os.path.basename(path))
        if m:
            ep = int(m.group(1))
            if ep > latest_epoch:
                latest_epoch, latest_path = ep, path
    return latest_epoch, latest_path

history = {
    "epoch": [],
    "train_loss": [], "test_loss": [],
    "train_mAP": [], "test_mAP": [],
}

max_mAP = 0.0
best_ckpt_path = None
start_epoch = 1

latest_epoch, latest_ckpt_path = _find_latest_checkpoint(RESUME_CKPT_DIR)
if latest_ckpt_path is not None:
    print(f"Found existing checkpoint: {latest_ckpt_path} (epoch {latest_epoch}) -- resuming training.")
    checkpoint = torch.load(latest_ckpt_path, map_location=device)
    model.load_state_dict(checkpoint["state_dict"])
    criterion.load_state_dict(checkpoint["criterion_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])
    history = checkpoint.get("history", history)
    max_mAP = checkpoint.get("max_mAP", 0.0)
    best_ckpt_path = checkpoint.get("best_ckpt_path", None)
    start_epoch = checkpoint["epoch"] + 1
    print(f"Resuming from epoch {start_epoch} (last completed epoch was {checkpoint['epoch']}).")
else:
    print("No existing checkpoint found in", RESUME_CKPT_DIR, "-- starting training from epoch 1.")

if start_epoch > args.epochs:
    print(f"Checkpoint already covers {start_epoch - 1} epoch(s), which meets or exceeds "
          f"args.epochs={args.epochs}. Nothing left to train.")

try:
    for epoch in range(start_epoch, args.epochs + 1):
        print_log(args.save_path, "----- %s at epoch #%d" % ("Train", epoch))
        train_log = train_one_epoch(args, train_dataset, train_loader, model, criterion, optimizer, epoch, device)
        scheduler.step()

        print_log(args.save_path, "----- %s at epoch #%d" % ("Test", epoch))
        test_log = test_one_epoch(args, test_dataset, test_loader, model, criterion, optimizer, epoch, device)

        history["epoch"].append(epoch)
        history["train_loss"].append(train_log.get("loss", float("nan")))
        history["test_loss"].append(test_log.get("loss", float("nan")))
        history["train_mAP"].append(train_log.get("mAP_train", 0.0))
        history["test_mAP"].append(test_log.get("mAP_test", 0.0))

        print(f"[Epoch {epoch}/{args.epochs}] "
              f"train_loss={history['train_loss'][-1]:.4f}  test_loss={history['test_loss'][-1]:.4f}  "
              f"train_mAP={history['train_mAP'][-1]:.2f}  test_mAP={history['test_mAP'][-1]:.2f}")

        if max_mAP < test_log["mAP_test"]:
            best_state = {
                "epoch": epoch,
                "state_dict": model.state_dict(),
                "criterion_dict": criterion.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
            }
            # remove any previous best checkpoint from this run
            for f in os.listdir(args.save_path):
                if f.endswith(".pth"):
                    os.remove(os.path.join(args.save_path, f))
            best_ckpt_path = os.path.join(args.save_path, "best_epoch%d.pth" % epoch)
            torch.save(best_state, best_ckpt_path)
            max_mAP = test_log["mAP_test"]
            print(f"  -> New best checkpoint saved: {best_ckpt_path} (test mAP={max_mAP:.2f})")

        # ---- resumable per-epoch checkpoint (always written, every epoch) ----
        epoch_state = {
            "epoch": epoch,
            "state_dict": model.state_dict(),
            "criterion_dict": criterion.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "history": history,
            "max_mAP": max_mAP,
            "best_ckpt_path": best_ckpt_path,
        }
        epoch_ckpt_path = os.path.join(RESUME_CKPT_DIR, f"epoch{epoch}.pt")
        _tmp_path = epoch_ckpt_path + ".tmp"
        torch.save(epoch_state, _tmp_path)
        os.replace(_tmp_path, epoch_ckpt_path)  # atomic rename: no half-written checkpoint on interruption
        print(f"  -> Saved resumable checkpoint: {epoch_ckpt_path}")

    print("\nTraining complete. Best test mAP:", max_mAP)
    print("Best checkpoint:", best_ckpt_path)
finally:
    # always restore stdout/stderr and close the log file, even if training raises
    sys.stdout, sys.stderr = _stdout_orig, _stderr_orig
    _log_file.close()
    print(f"Training log saved to: {log_file_path}")


## 10. Training curves

In [ ]:

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(history["epoch"], history["train_loss"], marker="o", label="train loss")
axes[0].plot(history["epoch"], history["test_loss"], marker="o", label="test loss")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history["epoch"], history["train_mAP"], marker="o", label="train mAP")
axes[1].plot(history["epoch"], history["test_mAP"], marker="o", label="test mAP")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("mAP@[0.3:0.7] (%)")
axes[1].set_title("Mean Average Precision")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(args.save_path, "training_curves.png"), dpi=150)
plt.show()


## 11. Final evaluation on the test set
Runs the repo's full evaluation pipeline (`on_tal_task.eval`, wired through `main.py --mode eval` in the README)
on the best checkpoint saved above, reporting mAP at each IoU threshold in `[0.3, 0.4, 0.5, 0.6, 0.7]`.

In [ ]:

import numpy as np
import util.misc as utils
from util.utils import parrallel_collate_fn, memory_initialize, online_nms, make_txt
from eval import evaluation_detection
import util.logger as loggers
import json

@torch.no_grad()
def run_final_evaluation(args, model, criterion, test_dataset, test_loader, device, ckpt_path):
    if ckpt_path is not None and os.path.exists(ckpt_path):
        checkpoint = torch.load(ckpt_path, map_location=device)
        pretrained_dict = {k: v for k, v in checkpoint["state_dict"].items() if k in model.state_dict()}
        model.load_state_dict(pretrained_dict)
        epoch = checkpoint["epoch"]
    else:
        epoch = args.epochs

    model.eval()
    criterion.eval()
    memory_initialize(model, args)

    metric_logger = loggers.MetricLogger(mode="evaluation", delimiter="  ")
    proposal_file = args.proposal_path.format({}, "finaleval", str(epoch))
    proposal_txt_path = os.path.join(args.save_path, proposal_file + ".txt")

    for i, (inputs, targets, infos) in enumerate(test_loader):
        inputs, targets, infos = parrallel_collate_fn(inputs, targets, infos, args.p_videos)
        inputs = inputs.to(device)
        targets = {k: v.to(device) for k, v in targets.items()}
        inputs = {"inputs": inputs, "infos": infos}
        outputs = model(inputs, device)
        if args.make_output:
            make_txt(args, infos, outputs, proposal_txt_path, test_dataset.label_name)

    proposal_json_path = os.path.join(args.save_path, proposal_file + ".json").format("pred")
    proposal_pred_txt_path = proposal_txt_path.format("pred")
    result_dict = online_nms(args, proposal_pred_txt_path, test_dataset)
    output_dict = {"version": "VERSION 1", "results": result_dict, "external_data": {}}
    with open(proposal_json_path, "w") as f:
        json.dump(output_dict, f, indent=2)

    tiou_thresholds = np.linspace(0.3, 0.70, 5)
    mAP = evaluation_detection(args, proposal_json_path, subset="test",
                                tiou_thresholds=tiou_thresholds, verbose=True)
    return mAP, proposal_json_path

final_mAP, final_proposal_json = run_final_evaluation(
    args, model, criterion, test_dataset, test_loader, device, best_ckpt_path
)

print("\n=== Final test-set mAP ===")
for thr, ap in zip([0.3, 0.4, 0.5, 0.6, 0.7], final_mAP):
    print(f"  mAP@{thr:.1f} = {ap:.2f}")
print(f"  Average mAP  = {final_mAP.mean():.2f}")


## 12. Inference example
Loads the predicted-proposals JSON produced above and prints the top predicted action instances
(label, start/end time in seconds, confidence score) for one sample video from the test split — this mirrors
what `make_txt` + `online_nms` (`util/utils.py`) produce at inference time for a streaming video.

In [ ]:

with open(final_proposal_json) as f:
    results = json.load(f)["results"]

sample_video = next(iter(results.keys()))
proposals = sorted(results[sample_video], key=lambda p: p["score"], reverse=True)

print(f"Sample video: {sample_video}")
print(f"Top predicted action instances (of {len(proposals)} total proposals):\n")
print(f"{'label':25s} {'start (s)':>10s} {'end (s)':>10s} {'score':>8s}")
for p in proposals[:10]:
    print(f"{p['label']:25s} {p['segment'][0]:10.2f} {p['segment'][1]:10.2f} {p['score']:8.3f}")


## 13. Download the trained checkpoint
The best checkpoint and training curve are saved under `args.save_path`. On Colab, the cell below zips them
for easy download; on Kaggle, everything under `/kaggle/working` is automatically included in the notebook's
output. No credentials or hardcoded local paths are used anywhere in this notebook.

In [ ]:

import shutil

zip_base = os.path.join(ROOT_DIR, "matr_training_outputs")
zip_path = shutil.make_archive(zip_base, "zip", args.save_path)
print("Packaged outputs:", zip_path)

if ENV == "colab":
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception as e:
        print("Automatic browser download not available in this context:", e)
        print("You can find the zipped outputs at:", zip_path)
else:
    print("Find your outputs at:", zip_path)
